### Scenario 3: Multiple data scientists working on multiple ML models

MLflow setup:

* Tracking server: yes, remote server (EC2).
* Backend store: postgresql database.
* Artifacts store: s3 bucket.
The experiments can be explored by accessing the remote server.

In [14]:
import mlflow
import os
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
import pandas as pd
import pickle
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import root_mean_squared_error
import mlflow
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope
import xgboost as xgb
from mlflow.tracking import MlflowClient

os.environ["AWS_PROFILE"] = "user1"

In [2]:
TRACKING_SERVER_HOST = "ec2-52-59-245-237.eu-central-1.compute.amazonaws.com"
mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:5000")
print(f"tracking URI: '{mlflow.get_tracking_uri()}'")

tracking URI: 'http://ec2-52-59-245-237.eu-central-1.compute.amazonaws.com:5000'


In [9]:
mlflow.search_experiments()

In [3]:
mlflow.set_experiment("iris-experiment")

with mlflow.start_run():

    X, y = load_iris(return_X_y=True)

    params = {"C": 0.1, "random_state": 42}
    mlflow.log_params(params)

    lr = LogisticRegression(**params).fit(X, y)
    y_pred = lr.predict(X)
    mlflow.log_metric("accuracy", accuracy_score(y, y_pred))

    mlflow.sklearn.log_model(lr, artifact_path="models")
    print(f"default artifacts URI: '{mlflow.get_artifact_uri()}'")

default artifacts URI: 's3://mlflow-artifact-remote11/1/4399ff353b614bc69559e554d64f2202/artifacts'


In [4]:
mlflow.search_experiments()

[<Experiment: artifact_location='s3://mlflow-artifact-remote11/1', creation_time=1720218364793, experiment_id='1', last_update_time=1720218364793, lifecycle_stage='active', name='iris-experiment', tags={}>,
 <Experiment: artifact_location='s3://mlflow-artifact-remote11/0', creation_time=1720196089468, experiment_id='0', last_update_time=1720196089468, lifecycle_stage='active', name='Default', tags={}>]

## NYC Taxi data

### Load data

In [8]:
def read_dataframe(filename):
    if filename.endswith('.csv'):
        df = pd.read_csv(filename)

        df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
        df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)
    elif filename.endswith('.parquet'):
        df = pd.read_parquet(filename)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df

df_train = read_dataframe('green_tripdata_2021-01.parquet')
df_val = read_dataframe('green_tripdata_2021-02.parquet')
len(df_train), len(df_val)


(73908, 61921)

### Prepare data

In [9]:
df_train['PU_DO'] = df_train['PULocationID'] + '_' + df_train['DOLocationID']
df_val['PU_DO'] = df_val['PULocationID'] + '_' + df_val['DOLocationID']
categorical = ['PU_DO'] #'PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [10]:
train = xgb.DMatrix(X_train, y_train)
valid = xgb.DMatrix(X_val, y_val)

def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("model", "xgboost")
        mlflow.log_params(params)
        booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=1000,
            evals=[(valid, 'validation')],
            early_stopping_rounds=50
        )
        y_pred = booster.predict(valid)
        rmse = root_mean_squared_error(y_val, y_pred)
        mlflow.log_metric("rmse", rmse)

    return {'loss': rmse, 'status': STATUS_OK}

In [ ]:
mlflow.set_experiment("nyc-taxi-experiment")
search_space = {
    'max_depth': scope.int(hp.quniform('max_depth', 4, 100, 1)),
    'learning_rate': hp.loguniform('learning_rate', -3, 0),
    'reg_alpha': hp.loguniform('reg_alpha', -5, -1),
    'reg_lambda': hp.loguniform('reg_lambda', -6, -1),
    'min_child_weight': hp.loguniform('min_child_weight', -1, 3),
    'objective': 'reg:linear',
    'seed': 42
}

best_result = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=20,
    trials=Trials()
)

### Training best model

In [12]:
best_params = {
    'learning_rate': 0.1574207155670002,
    'max_depth': 58,
    'min_child_weight': 1.056923368854001,
    'objective': 'reg:linear',
    'reg_alpha': 0.06532106054994348,  
    'reg_lambda': 0.08947561511697785,
    'seed': 42
}


with mlflow.start_run():
        mlflow.set_tag("model", "xgboost-final")
        mlflow.log_params(best_params)
        booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=1000,
            evals=[(valid, 'validation')],
            early_stopping_rounds=50
        )
        y_pred = booster.predict(valid)
        rmse = root_mean_squared_error(y_val, y_pred)

        with open("models/preprocessor.b", "wb") as f_out:
            pickle.dump(dv, f_out)

        mlflow.log_metric("rmse", rmse)
        mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")
        mlflow.xgboost.log_model(booster, artifact_path="models")

[0]	validation-rmse:9.96598
[1]	validation-rmse:8.60093
[2]	validation-rmse:7.81164
[3]	validation-rmse:7.36373
[4]	validation-rmse:7.11613
[5]	validation-rmse:6.97140
[6]	validation-rmse:6.88685
[7]	validation-rmse:6.83701
[8]	validation-rmse:6.80557
[9]	validation-rmse:6.78603


c:\Users\urbii\Desktop\Projekty\mlops-zoomcamp\01-Intro\venvi\Lib\site-packages\xgboost\core.py:158: UserWarning: [14:02:53] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-06abd128ca6c1688d-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "C" } are not used.

  warnings.warn(smsg, UserWarning)


[10]	validation-rmse:6.77386
[11]	validation-rmse:6.76695
[12]	validation-rmse:6.75853
[13]	validation-rmse:6.75421
[14]	validation-rmse:6.75023
[15]	validation-rmse:6.74823
[16]	validation-rmse:6.74648
[17]	validation-rmse:6.74455
[18]	validation-rmse:6.74144
[19]	validation-rmse:6.73458
[20]	validation-rmse:6.73161
[21]	validation-rmse:6.73012
[22]	validation-rmse:6.72696
[23]	validation-rmse:6.72602
[24]	validation-rmse:6.72392
[25]	validation-rmse:6.72251
[26]	validation-rmse:6.72081
[27]	validation-rmse:6.71992
[28]	validation-rmse:6.71750
[29]	validation-rmse:6.71560
[30]	validation-rmse:6.71401
[31]	validation-rmse:6.71145
[32]	validation-rmse:6.71019
[33]	validation-rmse:6.70887
[34]	validation-rmse:6.70784
[35]	validation-rmse:6.70616
[36]	validation-rmse:6.70606
[37]	validation-rmse:6.70411
[38]	validation-rmse:6.70163
[39]	validation-rmse:6.70035
[40]	validation-rmse:6.69939
[41]	validation-rmse:6.69882
[42]	validation-rmse:6.69725
[43]	validation-rmse:6.69505
[44]	validatio

c:\Users\urbii\Desktop\Projekty\mlops-zoomcamp\01-Intro\venvi\Lib\site-packages\xgboost\core.py:158: UserWarning: [14:03:07] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-06abd128ca6c1688d-1\xgboost\xgboost-ci-windows\src\c_api\c_api.cc:1374: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  warnings.warn(smsg, UserWarning)


### Register best model

In [16]:
client = MlflowClient(f"http://{TRACKING_SERVER_HOST}:5000")
client.search_registered_models()

[]

In [27]:
client.search_runs(experiment_ids=2)[0].data.tags['mlflow.log-model.history']

'[{"run_id": "1733bcf5fc484bdfb50461b0182158fd", "artifact_path": "models", "utc_time_created": "2024-07-06 12:03:07.535332", "flavors": {"python_function": {"loader_module": "mlflow.xgboost", "python_version": "3.12.4", "data": "model.xgb", "env": {"conda": "conda.yaml", "virtualenv": "python_env.yaml"}}, "xgboost": {"xgb_version": "2.1.0", "data": "model.xgb", "model_class": "xgboost.core.Booster", "model_format": "xgb", "code": null}}, "model_uuid": "1103526d6a1e47e7a7c4bc69b4e5714e", "mlflow_version": "2.14.1", "model_size_bytes": 1451809}]'

In [28]:
run_id = '1733bcf5fc484bdfb50461b0182158fd'
mlflow.register_model(
    model_uri=f"runs:/{run_id}/models",
    name='xgboost-regressor'
)

Successfully registered model 'xgboost-regressor'.
2024/07/06 14:09:09 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: xgboost-regressor, version 1
Created version '1' of model 'xgboost-regressor'.


<ModelVersion: aliases=[], creation_timestamp=1720267748544, current_stage='None', description='', last_updated_timestamp=1720267748544, name='xgboost-regressor', run_id='1733bcf5fc484bdfb50461b0182158fd', run_link='', source='s3://mlflow-artifact-remote11/2/1733bcf5fc484bdfb50461b0182158fd/artifacts/models', status='READY', status_message='', tags={}, user_id='', version='1'>

In [29]:
client.search_registered_models()

[<RegisteredModel: aliases={}, creation_timestamp=1720267748372, description='', last_updated_timestamp=1720267748544, latest_versions=[<ModelVersion: aliases=[], creation_timestamp=1720267748544, current_stage='None', description='', last_updated_timestamp=1720267748544, name='xgboost-regressor', run_id='1733bcf5fc484bdfb50461b0182158fd', run_link='', source='s3://mlflow-artifact-remote11/2/1733bcf5fc484bdfb50461b0182158fd/artifacts/models', status='READY', status_message='', tags={}, user_id='', version='1'>], name='xgboost-regressor', tags={}>]